In [7]:
import datasets

from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from transformers import GPT2Tokenizer, GPT2LMHeadModel, BartTokenizer, BartForConditionalGeneration
from transformers import get_linear_schedule_with_warmup

In [4]:
class WebNLGDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=128):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        # Format triples as text
        input_text = ' | '.join([f"{s} {p} {o}" for s, p, o in item['triples']])
        target_text = item['text']
        
        # Add special tokens
        input_text = f"Triple: {input_text} Text:"
        
        # Tokenize
        inputs = self.tokenizer(
            input_text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        targets = self.tokenizer(
            target_text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': inputs['input_ids'].squeeze(),
            'attention_mask': inputs['attention_mask'].squeeze(),
            'labels': targets['input_ids'].squeeze()
        }

In [5]:
class XSUMDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=512):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        
        inputs = self.tokenizer(
            item['document'],
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        targets = self.tokenizer(
            item['summary'],
            max_length=self.max_length//4,  # Shorter for summaries
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': inputs['input_ids'].squeeze(),
            'attention_mask': inputs['attention_mask'].squeeze(),
            'labels': targets['input_ids'].squeeze()
        }

In [ ]:
def load_webnlg():
    dataset = datasets.load_dataset('web_nlg', 'release_v2')
    return dataset

def load_xsum():
    dataset = datasets.load_dataset('xsum')
    return dataset

def prepare_dataloaders(dataset, tokenizer, batch_size=16, task='webnlg'):
    DatasetClass = WebNLGDataset if task == 'webnlg' else XSUMDataset
    
    train_dataset = DatasetClass(dataset['train'], tokenizer)
    val_dataset = DatasetClass(dataset['validation'], tokenizer)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)
    
    return train_loader, val_loader

In [ ]:
import torch
import torch.nn as nn

class GPTPrefixTuning(nn.Module):
    def __init__(
        self,
        base_model,
        prefix_length=10,
        prefix_hidden_size=512,
        use_projection=True,
        initialize_from_vocab=True
    ):
        super().__init__()
        self.base_model = base_model
        self.prefix_length = prefix_length
        self.hidden_size = base_model.config.n_embd
        
        # Freeze the base model
        for param in self.base_model.parameters():
            param.requires_grad = False
            
        if initialize_from_vocab:
            # Initialize from task-specific vocabulary
            init_tokens = ['table', 'to', 'text', 'generate', 'description']
            token_ids = torch.tensor(
                self.base_model.tokenizer.convert_tokens_to_ids(init_tokens)
            )
            token_embeds = self.base_model.get_input_embeddings()(token_ids)
            self.prefix_tokens = nn.Parameter(token_embeds)
        else:
            # Use projection network for prefix parameters
            if use_projection:
                self.prefix_projection = nn.Sequential(
                    nn.Linear(prefix_hidden_size, self.hidden_size),
                    nn.Tanh(),
                    nn.Linear(self.hidden_size, self.prefix_length * self.hidden_size)
                )
                self.prefix_embedding = nn.Embedding(prefix_length, prefix_hidden_size)
            else:
                self.prefix_tokens = nn.Parameter(
                    torch.randn(prefix_length, self.hidden_size)
                )
                self.prefix_projection = None
                
    def get_prefix(self, batch_size):
        if hasattr(self, 'prefix_projection') and self.prefix_projection is not None:
            prefix_tokens = self.prefix_embedding(
                torch.arange(self.prefix_length).to(self.prefix_embedding.weight.device)
            )
            prefix = self.prefix_projection(prefix_tokens)
            prefix = prefix.view(self.prefix_length, self.hidden_size)
        else:
            prefix = self.prefix_tokens
            
        return prefix.unsqueeze(0).expand(batch_size, -1, -1)
    
    def forward(self, input_ids, attention_mask=None, labels=None, **kwargs):
        batch_size = input_ids.shape[0]
        device = input_ids.device
        
        # Get prefix embeddings
        prefix = self.get_prefix(batch_size).to(device)
        
        # Get input embeddings
        inputs_embeds = self.base_model.get_input_embeddings()(input_ids)
        
        # Concatenate prefix and input embeddings
        inputs_embeds = torch.cat([prefix, inputs_embeds], dim=1)
        
        # Adjust attention mask for prefix
        if attention_mask is not None:
            prefix_mask = torch.ones(batch_size, self.prefix_length).to(device)
            attention_mask = torch.cat([prefix_mask, attention_mask], dim=1)
            
        # Forward pass through GPT
        outputs = self.base_model(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            labels=labels,
            **kwargs
        )
        
        return outputs


class BARTPrefixTuning(nn.Module):
    def __init__(
        self,
        base_model,
        prefix_length=200,  # Longer prefix for summarization
        prefix_hidden_size=512,
        use_projection=True,
        initialize_from_vocab=True
    ):
        super().__init__()
        self.base_model = base_model
        self.prefix_length = prefix_length
        self.hidden_size = base_model.config.hidden_size
        
        # Freeze the base model
        for param in self.base_model.parameters():
            param.requires_grad = False
            
        # Initialize prefix parameters for encoder and decoder
        if initialize_from_vocab:
            # Initialize from task-specific vocabulary
            init_tokens = ['summarize', 'brief', 'concise', 'summary', 'short']
            token_ids = torch.tensor(
                self.base_model.tokenizer.convert_tokens_to_ids(init_tokens)
            )
            token_embeds = self.base_model.get_input_embeddings()(token_ids)
            self.encoder_prefix = nn.Parameter(token_embeds.clone())
            self.decoder_prefix = nn.Parameter(token_embeds.clone())
        else:
            if use_projection:
                # Separate projections for encoder and decoder
                self.encoder_projection = nn.Sequential(
                    nn.Linear(prefix_hidden_size, self.hidden_size),
                    nn.Tanh(),
                    nn.Linear(self.hidden_size, self.prefix_length * self.hidden_size)
                )
                self.decoder_projection = nn.Sequential(
                    nn.Linear(prefix_hidden_size, self.hidden_size),
                    nn.Tanh(),
                    nn.Linear(self.hidden_size, self.prefix_length * self.hidden_size)
                )
                self.prefix_embedding = nn.Embedding(prefix_length, prefix_hidden_size)
            else:
                self.encoder_prefix = nn.Parameter(
                    torch.randn(prefix_length, self.hidden_size)
                )
                self.decoder_prefix = nn.Parameter(
                    torch.randn(prefix_length, self.hidden_size)
                )
                self.encoder_projection = None
                self.decoder_projection = None
                
    def get_prefix(self, batch_size, encoder=True):
        if hasattr(self, 'encoder_projection') and self.encoder_projection is not None:
            prefix_tokens = self.prefix_embedding(
                torch.arange(self.prefix_length).to(self.prefix_embedding.weight.device)
            )
            if encoder:
                prefix = self.encoder_projection(prefix_tokens)
            else:
                prefix = self.decoder_projection(prefix_tokens)
            prefix = prefix.view(self.prefix_length, self.hidden_size)
        else:
            prefix = self.encoder_prefix if encoder else self.decoder_prefix
            
        return prefix.unsqueeze(0).expand(batch_size, -1, -1)
    
    def forward(self, input_ids, attention_mask=None, labels=None, **kwargs):
        batch_size = input_ids.shape[0]
        device = input_ids.device
        
        # Get encoder prefix
        encoder_prefix = self.get_prefix(batch_size, encoder=True).to(device)
        
        # Encoder forward pass
        encoder_outputs = self.base_model.get_encoder()(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True
        )
        
        # Prepend prefix to encoder outputs
        encoder_hidden_states = torch.cat(
            [encoder_prefix, encoder_outputs.last_hidden_state], 
            dim=1
        )
        
        # Adjust attention mask for prefix
        if attention_mask is not None:
            prefix_mask = torch.ones(batch_size, self.prefix_length).to(device)
            attention_mask = torch.cat([prefix_mask, attention_mask], dim=1)
            
        # Get decoder prefix
        decoder_prefix = self.get_prefix(batch_size, encoder=False).to(device)
        
        # Forward pass through BART
        outputs = self.base_model(
            encoder_outputs=encoder_hidden_states,
            attention_mask=attention_mask,
            decoder_inputs_embeds=decoder_prefix if labels is None else None,
            labels=labels,
            **kwargs
        )
        
        return outputs

In [ ]:
def train_webnlg():
    """
    Setup and train GPT model with prefix tuning for WebNLG dataset
    """
    # Initialize models and tokenizers
    tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
    base_model = GPT2LMHeadModel.from_pretrained('gpt2')
    
    # Add special tokens
    special_tokens = {
        'pad_token': '<|pad|>', 
        'sep_token': '<|sep|>',
        'bos_token': '<|bos|>',
        'eos_token': '<|eos|>'
    }
    tokenizer.add_special_tokens(special_tokens)
    base_model.resize_token_embeddings(len(tokenizer))
    
    # Attach tokenizer to base model for initialization
    base_model.tokenizer = tokenizer
    
    # Create prefix-tuning model
    model = GPTPrefixTuning(
        base_model=base_model,
        prefix_length=10,
        prefix_hidden_size=512,
        use_projection=True,
        initialize_from_vocab=True
    )
    
    # Load data
    dataset = load_webnlg()
    train_loader, val_loader = prepare_dataloaders(
        dataset, 
        tokenizer, 
        task='webnlg',
        batch_size=16,
        max_length=128
    )
    
    return model, train_loader, val_loader, tokenizer

def train_xsum():
    """
    Setup and train BART model with prefix tuning for XSUM dataset
    """
    # Initialize models and tokenizers
    tokenizer = BartTokenizer.from_pretrained('facebook/bart-large')
    base_model = BartForConditionalGeneration.from_pretrained('facebook/bart-large')
    
    # Attach tokenizer to base model for initialization
    base_model.tokenizer = tokenizer
    
    # Create prefix-tuning model
    model = BARTPrefixTuning(
        base_model=base_model,
        prefix_length=200,  # Longer prefix for summarization
        prefix_hidden_size=512,
        use_projection=True,
        initialize_from_vocab=True
    )
    
    # Load data
    dataset = load_xsum()
    train_loader, val_loader = prepare_dataloaders(
        dataset, 
        tokenizer, 
        task='summarization',
        batch_size=8,  # Smaller batch size due to longer sequences
        max_length=512
    )
    
    return model, train_loader, val_loader, tokenizer

def train_and_evaluate(
    model,
    train_loader,
    val_loader,
    tokenizer,
    num_epochs=10,
    device='cuda',
    task='webnlg',
    gradient_accumulation_steps=1,
    max_grad_norm=1.0,
    warmup_steps=1000
):
    """
    Enhanced training loop with better optimization and monitoring
    """
    model = model.to(device)
    
    # Setup optimizer with weight decay
    no_decay = ['bias', 'LayerNorm.weight']
    optimizer_grouped_parameters = [
        {
            'params': [p for n, p in model.named_parameters() 
                      if not any(nd in n for nd in no_decay)],
            'weight_decay': 0.01
        },
        {
            'params': [p for n, p in model.named_parameters() 
                      if any(nd in n for nd in no_decay)],
            'weight_decay': 0.0
        }
    ]
    
    optimizer = torch.optim.AdamW(optimizer_grouped_parameters, lr=5e-5)
    
    # Setup learning rate scheduler
    total_steps = len(train_loader) * num_epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )
    
    best_val_loss = float('inf')
    training_stats = []
    
    for epoch in range(num_epochs):
        print(f'\nEpoch {epoch+1}/{num_epochs}')
        print('-' * 40)
        
        # Training phase
        model.train()
        total_train_loss = 0
        steps = 0
        
        for batch_idx, batch in enumerate(train_loader):
            batch = {k: v.to(device) for k, v in batch.items()}
            
            outputs = model(**batch)
            loss = outputs.loss / gradient_accumulation_steps
            loss.backward()
            
            total_train_loss += loss.item()
            
            # Gradient accumulation
            if (batch_idx + 1) % gradient_accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                steps += 1
                
                # Progress update
                if steps % 100 == 0:
                    print(f'Batch {batch_idx+1}/{len(train_loader)} | '
                          f'Loss: {loss.item():.4f}')
        
        avg_train_loss = total_train_loss / len(train_loader)
        
        # Validation phase
        model.eval()
        total_val_loss = 0
        val_steps = 0
        
        print("\nRunning Validation...")
        
        with torch.no_grad():
            for batch in val_loader:
                batch = {k: v.to(device) for k, v in batch.items()}
                outputs = model(**batch)
                loss = outputs.loss
                total_val_loss += loss.item()
                val_steps += 1
                
        avg_val_loss = total_val_loss / val_steps
        
        # Store statistics
        training_stats.append({
            'epoch': epoch + 1,
            'training_loss': avg_train_loss,
            'validation_loss': avg_val_loss,
            'learning_rate': scheduler.get_last_lr()[0]
        })
        
        print(f'\nAverage training loss: {avg_train_loss:.4f}')
        print(f'Average validation loss: {avg_val_loss:.4f}')
        
        # Save best model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            model_path = f'best_model_{task}_epoch_{epoch+1}.pt'
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': best_val_loss,
            }, model_path)
            print(f"Saved best model to {model_path}")
    
    return training_stats

# Example usage
if __name__ == "__main__":
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Train WebNLG
    print("Training WebNLG model...")
    webnlg_model, webnlg_train, webnlg_val, webnlg_tokenizer = train_webnlg()
    webnlg_stats = train_and_evaluate(
        webnlg_model,
        webnlg_train,
        webnlg_val,
        webnlg_tokenizer,
        num_epochs=10,
        device=device,
        task='webnlg'
    )
    
    # Train XSUM
    print("\nTraining XSUM model...")
    xsum_model, xsum_train, xsum_val, xsum_tokenizer = train_xsum()
    xsum_stats = train_and_evaluate(
        xsum_model,
        xsum_train,
        xsum_val,
        xsum_tokenizer,
        num_epochs=10,
        device=device,
        task='summarization'
    )